# Práctica 5 · Recomendador de películas

Esta práctica usa las valoraciones de MovieLens Latest Small para entrenar un recomendador. El cuaderno incluye una comparación de modelos, recomendaciones para dos perfiles y un análisis de sesgos, películas cercanas y grupos de usuarios.

## Preparar el cuaderno

Usa Python 3.12. Crea el entorno con `python -m venv .venv`. En PowerShell, actívalo con `.\.venv\Scripts\Activate.ps1`. Instala las dependencias con `python -m pip install -r requirements.txt` y abre este archivo con `jupyter lab practica_movielens.ipynb`.

Guarda el ZIP de MovieLens en `.data/ml-latest-small.zip` y ejecuta las celdas en orden.

[MovieLens Latest Small](https://grouplens.org/datasets/movielens/latest/)

In [1]:
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
from IPython.display import display

SEMILLA = 42
COLUMNAS = ["userId", "movieId", "rating"]
ESCALA = (0.5, 5)
ruta_zip = Path(".data/ml-latest-small.zip")

# Lee las dos tablas del ZIP oficial
if not ruta_zip.is_file():
    raise FileNotFoundError(f"Guarda el ZIP de MovieLens en {ruta_zip}")

with ZipFile(ruta_zip) as archivo:
    valoraciones = pd.read_csv(archivo.open("ml-latest-small/ratings.csv"))
    peliculas = pd.read_csv(archivo.open("ml-latest-small/movies.csv"))

print("Valoraciones:", len(valoraciones))
print("Usuarios:", valoraciones["userId"].nunique())
print("Películas del catálogo:", len(peliculas))
display(valoraciones.head())

Valoraciones: 100836
Usuarios: 610
Películas del catálogo: 9742


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


El catálogo también contiene películas sin valoraciones. Para calcular la densidad se usan los usuarios y las películas que tienen al menos una nota.

In [2]:
notas_usuario = valoraciones.groupby("userId").size()
notas_pelicula = valoraciones.groupby("movieId").size()
densidad = len(valoraciones) / (len(notas_usuario) * len(notas_pelicula))

assert valoraciones["rating"].between(*ESCALA).all()
assert not valoraciones.duplicated(["userId", "movieId"]).any()

print("Películas con notas:", len(notas_pelicula))
print(f"Densidad de la matriz: {densidad:.2%}")
print("Mediana de notas por usuario:", notas_usuario.median())

Películas con notas: 9724
Densidad de la matriz: 1.70%
Mediana de notas por usuario: 70.5


## 1. Entrenamiento y evaluación

Se reserva el 20 % más reciente de las valoraciones de cada usuario para prueba. Si hay notas con la misma fecha, se ordenan por identificador de película. Este corte respeta el orden por usuario. Con un corte por fecha de calendario se mediría un escenario distinto.

In [3]:
from surprise import BaselineOnly, Dataset, Reader, SVD, accuracy

ordenadas = valoraciones.sort_values(["userId", "timestamp", "movieId"])
por_usuario = ordenadas.groupby("userId")
cantidad = por_usuario["rating"].transform("size")
notas_prueba = np.ceil(cantidad * 0.20)
# Deja en prueba las notas mas recientes de cada usuario
en_prueba = por_usuario.cumcount() >= cantidad - notas_prueba

lector = Reader(rating_scale=ESCALA)
entrenamiento = Dataset.load_from_df(ordenadas.loc[~en_prueba, COLUMNAS], lector)
entrenamiento = entrenamiento.build_full_trainset()
prueba = list(ordenadas.loc[en_prueba, COLUMNAS].itertuples(index=False, name=None))

print("Notas de entrenamiento:", entrenamiento.n_ratings)
print("Notas de prueba:", len(prueba))

Notas de entrenamiento: 80419
Notas de prueba: 20417


MAE y RMSE miden el error en estrellas. RMSE da más peso a las desviaciones grandes. Se evalúan SVD de 20 factores y un modelo de media y sesgos con los mismos datos.

In [4]:
ajustes_svd = {"n_factors": 20, "random_state": SEMILLA}
referencia = BaselineOnly(verbose=False)
svd = SVD(**ajustes_svd)
resultados = []
for nombre, modelo in [("Media y sesgos", referencia), ("SVD", svd)]:
    modelo.fit(entrenamiento)
    predicciones = modelo.test(prueba)
    resultados.append({
        "Modelo": nombre,
        "MAE": accuracy.mae(predicciones, verbose=False),
        "RMSE": accuracy.rmse(predicciones, verbose=False),
    })

display(pd.DataFrame(resultados).round(4))

,Modelo,MAE,RMSE
0,Media y sesgos,0.6977,0.9024
1,SVD,0.6822,0.8861


El error obtenido describe la predicción de notas observadas. La calidad del orden de una lista de recomendaciones necesita una evaluación propia.

[SVD en Surprise](https://surprise.readthedocs.io/en/stable/matrix_factorization.html)

## 2. Recomendaciones para dos perfiles

Las notas de los perfiles son ejemplos editables. El perfil A puntúa más alto las películas de animación. El perfil B prefiere películas de crimen. Se incorporan los perfiles al conjunto completo y se vuelve a entrenar SVD. Para cada perfil se calculan las diez películas con mayor puntuación estimada entre las que aún no ha valorado.

In [5]:
perfiles = {
    "Perfil A": {1: 5, 2: 4, 6: 2, 50: 3, 296: 2.5, 318: 4, 588: 5, 595: 5},
    "Perfil B": {1: 3, 2: 2.5, 6: 5, 50: 5, 296: 5, 318: 4.5, 588: 2, 595: 2.5},
}
primer_id = valoraciones["userId"].max() + 1
ids_perfiles = dict(zip(perfiles, range(primer_id, primer_id + len(perfiles))))
filas_nuevas = []
for nombre, notas in perfiles.items():
    for pelicula, nota in notas.items():
        filas_nuevas.append((ids_perfiles[nombre], pelicula, nota))

nuevas = pd.DataFrame(filas_nuevas, columns=COLUMNAS)
if not nuevas["movieId"].isin(notas_pelicula.index).all():
    raise ValueError("Los perfiles deben usar películas con valoraciones en MovieLens.")
if not nuevas["rating"].between(*ESCALA).all():
    raise ValueError("Las notas de los perfiles deben estar entre 0,5 y 5.")
display(nuevas.merge(peliculas[["movieId", "title"]], on="movieId"))

,userId,movieId,rating,title
0,611,1,5.0,Toy Story (1995)
1,611,2,4.0,Jumanji (1995)
2,611,6,2.0,Heat (1995)
3,611,50,3.0,"Usual Suspects, The (1995)"
4,611,296,2.5,Pulp Fiction (1994)
5,611,318,4.0,"Shawshank Redemption, The (1994)"
6,611,588,5.0,Aladdin (1992)
7,611,595,5.0,Beauty and the Beast (1991)
8,612,1,3.0,Toy Story (1995)
9,612,2,2.5,Jumanji (1995)
